# Notebook 06: Plagiarism Detection

This notebook compares student answers within a batch using sentence embeddings and flags suspiciously similar answers according to the project thresholds.

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    REPO_ROOT = Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_MODEL = SentenceTransformer(MODEL_NAME)
print(f"Loaded embedding model: {MODEL_NAME}")

In [ ]:
CLEANED_JSON = REPO_ROOT / "outputs" / "cleaned_qa.json"
if not CLEANED_JSON.exists():
    raise FileNotFoundError(f"Cleaned QA file not found: {CLEANED_JSON}. Run the NLP notebook first.")

rows = json.loads(CLEANED_JSON.read_text(encoding="utf-8"))
print(f"Loaded {len(rows)} cleaned answer rows")

In [ ]:
from collections import defaultdict

question_groups = defaultdict(list)
for row in rows:
    question_groups[row["q_no"]].append(row)

print(f"Grouped into {len(question_groups)} question(s)")

In [ ]:
results = []

for q_no, items in question_groups.items():
    if len(items) < 2:
        continue

    answers = [item["cleaned_answer"] for item in items if item.get("cleaned_answer")]
    if len(answers) < 2:
        continue

    embeddings = EMBEDDING_MODEL.encode(answers, convert_to_tensor=True)
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            score = torch.nn.functional.cosine_similarity(
                embeddings[i].unsqueeze(0),
                embeddings[j].unsqueeze(0),
            ).item()
            percent = round(score * 100, 2)
            if percent >= 92:
                if percent >= 96:
                    status = "HIGH_RISK"
                else:
                    status = "SUSPICIOUS"
                results.append({
                    "question_no": q_no,
                    "student_a": items[i].get("image", "student_1"),
                    "student_b": items[j].get("image", "student_2"),
                    "similarity_percent": percent,
                    "status": status,
                })

results_path = REPO_ROOT / "outputs" / "plagiarism_results.json"
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Saved {len(results)} suspicious pair(s) to {results_path}")

for row in results[:10]:
    print(row)

## Interpretation

- Above 96% similarity: HIGH RISK
- 92% to 96% similarity: SUSPICIOUS
- Below 92%: normal